# 12 · Hybrid Retrieval, Reranking, and Citation Checks
**Time:** 60 minutes · **Prerequisites:** 09–10

The default lesson compares BM25 and TF-IDF offline. Set `RUN_DENSE = True`
and install the `embeddings` group for MiniLM, reciprocal-rank fusion (RRF),
and a cross-encoder. The runner enables this with `--advanced` and an explicit
model download budget. [Sentence Transformers' retrieve/rerank guide](https://www.sbert.net/examples/sparse_encoder/applications/retrieve_rerank/README.html)
provides the upstream pattern.

**Objectives:** compare rankings with one evaluation set; avoid mixing score
scales; measure reranking cost; preserve source IDs when changing chunk size;
distinguish a valid citation from an answer supported by its cited source.

In [ ]:
import time
from collections import Counter
from pathlib import Path
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from mlx_nlp_utils import load_rag_knowledge_base, load_rag_eval_queries, tokenize
from experiment_utils import rrf, retrieval_metrics, citation_metrics, model_path, write_record

knowledge_base = load_rag_knowledge_base()
examples = load_rag_eval_queries()
queries = [r['query'] for r in examples]
relevant = [{r['relevant_doc']} for r in examples]


## 1. BM25: lexical evidence with length normalization
BM25 saturates repeated term counts and compensates for document length.
Here k1=1.2 and b=0.75 are fixed teaching defaults, not tuned on the test queries.
The implementation is intentionally small so you can inspect every score.
Predict which exact identifiers a dense encoder might miss.

In [ ]:
def bm25(documents, queries, k1=1.2, b=0.75):
    counts = [Counter(tokenize(doc)) for doc in documents]
    lengths = np.array([sum(c.values()) for c in counts], dtype=float)
    average = max(float(lengths.mean()), 1.0)
    df = Counter(term for count in counts for term in count)
    scores = np.zeros((len(queries), len(documents)))
    for i, query in enumerate(queries):
        for term in set(tokenize(query)):
            freq = np.array([c[term] for c in counts])
            idf = np.log(1 + (len(documents) - df[term] + 0.5) / (df[term] + 0.5))
            scores[i] += idf * freq * (k1 + 1) / (freq + k1 * (1 - b + b * lengths / average))
    return scores

bm_scores = bm25(knowledge_base, queries)
vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(knowledge_base)
tf_scores = (vectorizer.transform(queries) @ doc_vectors.T).toarray()
rankings = {"BM25": np.argsort(-bm_scores, axis=1, kind='stable'),
            "TF-IDF": np.argsort(-tf_scores, axis=1, kind='stable')}
results = {name: retrieval_metrics(order, relevant) for name, order in rankings.items()}
print(results)


## 2. Dense candidates → fusion → reranking
RRF adds 1/(60+rank) from each ranked list. It does not assume BM25 and cosine
scores share units. A cross-encoder then jointly reads each query/document pair
for the top ten fused candidates. It cannot recover a document absent from
that candidate set. This corpus is small, so exact search is adequate.

In [ ]:
RUN_DENSE = False
models_used = []
if RUN_DENSE:
    from sentence_transformers import SentenceTransformer, CrossEncoder
    encoder = SentenceTransformer(model_path('minilm'))
    docs = encoder.encode_document(knowledge_base, normalize_embeddings=True)
    qs = encoder.encode_query(queries, normalize_embeddings=True)
    rankings['dense'] = np.argsort(-(qs @ docs.T), axis=1, kind='stable')
    rankings['RRF'] = np.array([rrf([lexical, dense]) for lexical, dense in
                               zip(rankings['BM25'], rankings['dense'])])
    reranker = CrossEncoder(model_path('reranker'))
    reranked = []
    start = time.perf_counter()
    for query, order in zip(queries, rankings['RRF']):
        candidates = order[:10]
        scores = reranker.predict([(query, knowledge_base[i]) for i in candidates])
        reranked.append(np.r_[candidates[np.argsort(-scores, kind='stable')], order[10:]])
    rankings['RRF + reranker'] = np.array(reranked)
    print('Rerank seconds (includes first-call warm-up):', time.perf_counter() - start)
    models_used = ['minilm', 'reranker']
    results.update({name: retrieval_metrics(order, relevant) for name, order in rankings.items()})
for name, metrics in results.items():
    print(name, metrics)


## 3. Chunk size changes relevance labels
Combine adjacent documents into chunks of 1, 2, or 5. A relevant source maps
to its containing chunk. Failing to remap the labels creates a bogus metric.
Recall can rise simply because a chunk carries more text; report retrieved
word counts too. This is a controlled grouping experiment, not a realistic
paragraph-boundary chunker. Tune chunk size on development queries before
reporting performance on a separate final set.

In [ ]:
chunk_results = []
for size in [1, 2, 5]:
    chunks = [' '.join(knowledge_base[i:i+size]) for i in range(0, len(knowledge_base), size)]
    chunk_gold = [{next(iter(gold)) // size} for gold in relevant]
    order = np.argsort(-bm25(chunks, queries), axis=1, kind='stable')
    words = np.mean([sum(len(chunks[i].split()) for i in row[:3]) for row in order])
    chunk_results.append({'documents_per_chunk':size, 'mean_retrieved_words':float(words),
                          **retrieval_metrics(order, chunk_gold)})
print(chunk_results)


## 4. A citation can be well formed and still wrong
First check that cited IDs exist in the retrieved context. Then check whether
a gold supporting source was cited. Neither check establishes that every
claim in a generated answer is entailed by its citation. This lesson uses
controlled citations so the failure is visible without an LLM download.
For generated answers, add claim-level human review and record disagreements.

In [ ]:
selected = rankings.get('RRF + reranker', rankings['BM25'])
query_index = 0
retrieved = selected[query_index][:3].tolist()
print('Question:', queries[query_index])
for doc_id in retrieved:
    print(f'[{doc_id}] {knowledge_base[doc_id]}')
for name, citations in [('top result', retrieved[:1]), ('fabricated ID', [999])]:
    print(name, citation_metrics(citations, retrieved, relevant[query_index]))
# Even a real source ID can be attached to a false statement.
print('A structurally valid citation is not proof of factual support.')
print(write_record('12-retrieval', models=models_used, seed=0,
    settings={'bm25_k1':1.2, 'bm25_b':0.75, 'rrf_k':60, 'rerank_candidates':10,
              'dense_enabled':RUN_DENSE}, metrics={'retrieval':results, 'chunking':chunk_results},
    data_files=[Path('../data/rag_samples/knowledge_base.json'), Path('../data/rag_samples/eval_queries.json')]))


## Exercises
1. Break down every method's metrics by lexical versus paraphrase queries.
2. Add exact product IDs and confusing near-matches, then compare lexical and dense retrieval.
3. Lower the reranking candidate count. Does candidate recall limit final recall?
4. Write supported and unsupported answers with identical citation IDs and
   assess them manually. Why does the structural checker accept both?

All 40 queries are a teaching evaluation set. Once you use them to choose a
retriever, collect an untouched test set before making performance claims.